# ARMA Parameter Recovery — Visualization

Loads the pre-trained SimpleModel (H1024) and the best DeepGRU recovery head,
then plots evaluation graphs. **No training required.**

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

from arma import generate_arma_batch
from network import SimpleModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

NUM_ARMA_PARAMS = 4

In [ ]:
# ── DeepGRU model definition (needed to load weights) ──

class DeepGRURecoveryHead(nn.Module):
    def __init__(self, H=1024, hidden_dim=256, num_arma_params=4, num_gru_layers=3):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(H, hidden_dim * 2), nn.SiLU(), nn.LayerNorm(hidden_dim * 2),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.SiLU(), nn.LayerNorm(hidden_dim),
        )
        self.gru = nn.GRU(input_size=hidden_dim, hidden_size=hidden_dim,
                          num_layers=num_gru_layers, batch_first=True,
                          dropout=0.1, bidirectional=True)
        gru_out = hidden_dim * 2
        self.mid_proj = nn.Sequential(
            nn.Linear(gru_out, hidden_dim), nn.SiLU(), nn.LayerNorm(hidden_dim),
        )
        self.res_block1 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2), nn.SiLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.res_block2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2), nn.SiLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.ar_heads = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 2), nn.SiLU(), nn.Linear(hidden_dim // 2, 1))
            for _ in range(num_arma_params)
        ])
        self.ma_heads = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 2), nn.SiLU(), nn.Linear(hidden_dim // 2, 1))
            for _ in range(num_arma_params)
        ])
        self.num_arma_params = num_arma_params

    def forward(self, x):
        x = self.input_proj(x)
        gru_out, _ = self.gru(x)
        x = self.mid_proj(gru_out)
        x = self.norm1(x + self.res_block1(x))
        x = self.norm2(x + self.res_block2(x))
        ar = torch.cat([h(x) for h in self.ar_heads], dim=-1)
        ma = torch.cat([h(x) for h in self.ma_heads], dim=-1)
        return torch.tanh(ar), torch.tanh(ma)

In [ ]:
# ── Load models ──

model = SimpleModel(C=4, H=1024, W=32, num_layers=12)
model.load_state_dict(torch.load('trained_simple_model_H1024.pth', map_location=device))
model = model.to(device)
model.eval()
for p in model.parameters():
    p.requires_grad = False
print('Contrastive backbone loaded (H1024, 2M steps)')

param_head = DeepGRURecoveryHead(H=1024, hidden_dim=256, num_arma_params=NUM_ARMA_PARAMS)
param_head.load_state_dict(torch.load('parameter_recovery_deepgru_best.pth', map_location=device))
param_head = param_head.to(device)
param_head.eval()
print('DeepGRU recovery head loaded')

In [ ]:
# ── Helpers ──

def extract_latent_features(model, x):
    with torch.no_grad():
        _, h = model(x)
        B, T, C, H = h.shape
        return h.permute(0, 2, 1, 3).reshape(B * C, T, H)

def prepare_data(batch_size=32, seed=None):
    x, parameters = generate_arma_batch(batch_size=batch_size, T_raw=4096, C=4, seed=seed, dimension=4)
    x = x.to(device)
    true_ar, true_ma = [], []
    for ar_poly, ma_poly in parameters:
        ar = np.pad(-ar_poly[1:], (0, max(0, NUM_ARMA_PARAMS - len(ar_poly) + 1)))[:NUM_ARMA_PARAMS]
        ma = np.pad( ma_poly[1:], (0, max(0, NUM_ARMA_PARAMS - len(ma_poly) + 1)))[:NUM_ARMA_PARAMS]
        true_ar.append(ar); true_ma.append(ma)
    return x, torch.tensor(np.array(true_ar), dtype=torch.float32, device=device), \
              torch.tensor(np.array(true_ma), dtype=torch.float32, device=device)

## 1. Evaluation metrics

In [ ]:
all_ar_err, all_ma_err, all_total_err, all_baseline = [], [], [], []

with torch.no_grad():
    for i in range(200):
        x, ar_true, ma_true = prepare_data(batch_size=1, seed=i + 1000)
        h = extract_latent_features(model, x)
        pred_ar, pred_ma = param_head(h)
        ar_e = F.mse_loss(pred_ar.mean(1), ar_true).item()
        ma_e = F.mse_loss(pred_ma.mean(1), ma_true).item()
        bl = F.mse_loss(torch.zeros_like(ar_true), ar_true).item() + \
             F.mse_loss(torch.zeros_like(ma_true), ma_true).item()
        all_ar_err.append(ar_e); all_ma_err.append(ma_e)
        all_total_err.append(ar_e + ma_e); all_baseline.append(bl)

print('=== Parameter Recovery Performance (DeepGRU h256) ===')
print(f'Mean AR Error:    {np.mean(all_ar_err):.6f} +/- {np.std(all_ar_err):.6f}')
print(f'Mean MA Error:    {np.mean(all_ma_err):.6f} +/- {np.std(all_ma_err):.6f}')
print(f'Mean Total Error: {np.mean(all_total_err):.6f} +/- {np.std(all_total_err):.6f}')
print(f'Baseline (zero):  {np.mean(all_baseline):.6f}')
print(f'Improvement:      {np.mean(all_baseline) / np.mean(all_total_err):.2f}x')

## 2. True vs Predicted parameters

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(12, 15))

with torch.no_grad():
    for i in range(5):
        x, ar_true, ma_true = prepare_data(batch_size=1, seed=i + 2000)
        h = extract_latent_features(model, x)
        pred_ar, pred_ma = param_head(h)
        ar_pred = pred_ar[0].mean(0).cpu().numpy()
        ma_pred = pred_ma[0].mean(0).cpu().numpy()
        ar_true_np = ar_true[0].cpu().numpy()
        ma_true_np = ma_true[0].cpu().numpy()

        idx = np.arange(NUM_ARMA_PARAMS)
        w = 0.35
        axes[i, 0].bar(idx - w/2, ar_true_np, w, label='True', color='steelblue')
        axes[i, 0].bar(idx + w/2, ar_pred,    w, label='Predicted', color='indianred')
        axes[i, 0].set_title(f'AR Parameters — Sample {i+1}')
        axes[i, 0].set_xlabel('Coefficient index')
        axes[i, 0].set_ylabel('Value')
        axes[i, 0].legend()
        axes[i, 0].grid(True, alpha=0.3)
        axes[i, 0].set_xticks(idx)

        axes[i, 1].bar(idx - w/2, ma_true_np, w, label='True', color='steelblue')
        axes[i, 1].bar(idx + w/2, ma_pred,    w, label='Predicted', color='indianred')
        axes[i, 1].set_title(f'MA Parameters — Sample {i+1}')
        axes[i, 1].set_xlabel('Coefficient index')
        axes[i, 1].set_ylabel('Value')
        axes[i, 1].legend()
        axes[i, 1].grid(True, alpha=0.3)
        axes[i, 1].set_xticks(idx)

plt.tight_layout()
plt.show()

## 3. Error distributions

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.hist(all_ar_err, bins=30, alpha=0.7, color='steelblue')
plt.axvline(np.mean(all_ar_err), color='red', ls='--', label=f'Mean: {np.mean(all_ar_err):.4f}')
plt.xlabel('AR Parameter MSE'); plt.ylabel('Frequency')
plt.title('AR Error Distribution'); plt.legend()

plt.subplot(1, 3, 2)
plt.hist(all_ma_err, bins=30, alpha=0.7, color='seagreen')
plt.axvline(np.mean(all_ma_err), color='red', ls='--', label=f'Mean: {np.mean(all_ma_err):.4f}')
plt.xlabel('MA Parameter MSE'); plt.ylabel('Frequency')
plt.title('MA Error Distribution'); plt.legend()

plt.subplot(1, 3, 3)
plt.hist(all_total_err, bins=30, alpha=0.7, color='mediumpurple')
plt.axvline(np.mean(all_total_err), color='red', ls='--', label=f'Mean: {np.mean(all_total_err):.4f}')
plt.xlabel('Total Parameter MSE'); plt.ylabel('Frequency')
plt.title('Total Error Distribution'); plt.legend()

plt.tight_layout()
plt.show()

## 4. Sign agreement analysis

In [ ]:
all_true_ar, all_pred_ar = [], []
all_true_ma, all_pred_ma = [], []

with torch.no_grad():
    for i in range(300):
        x, ar_true, ma_true = prepare_data(batch_size=1, seed=i + 1000)
        h = extract_latent_features(model, x)
        pred_ar, pred_ma = param_head(h)
        all_true_ar.append(ar_true.cpu().numpy())
        all_pred_ar.append(pred_ar.mean(1).cpu().numpy())
        all_true_ma.append(ma_true.cpu().numpy())
        all_pred_ma.append(pred_ma.mean(1).cpu().numpy())

true_ar = np.concatenate(all_true_ar)
pred_ar = np.concatenate(all_pred_ar)
true_ma = np.concatenate(all_true_ma)
pred_ma = np.concatenate(all_pred_ma)

threshold = 0.05
print('Per-coefficient sign agreement (|true| > 0.05):')
print(f'{"":4s} {"Sign":>8s} {"Corr":>8s} {"MAE":>8s}')
for name, true, pred in [('AR', true_ar, pred_ar), ('MA', true_ma, pred_ma)]:
    for j in range(NUM_ARMA_PARAMS):
        mask = np.abs(true[:, j]) > threshold
        if mask.sum() == 0: continue
        t, p = true[mask, j], pred[mask, j]
        sign = (np.sign(t) == np.sign(p)).mean()
        corr = np.corrcoef(t, p)[0, 1]
        mae = np.abs(t - p).mean()
        print(f'{name}[{j}] {sign:8.1%} {corr:8.3f} {mae:8.3f}')
    mask = np.abs(true) > threshold
    t_flat, p_flat = true[mask], pred[mask]
    sign_all = (np.sign(t_flat) == np.sign(p_flat)).mean()
    corr_all = np.corrcoef(t_flat, p_flat)[0, 1]
    print(f'{name} ALL {sign_all:7.1%} {corr_all:8.3f}')
    print()

## 5. Scatter plots — True vs Predicted

In [ ]:
fig, axes = plt.subplots(2, NUM_ARMA_PARAMS, figsize=(4 * NUM_ARMA_PARAMS, 8))

for j in range(NUM_ARMA_PARAMS):
    for row, (name, true, pred) in enumerate([('AR', true_ar, pred_ar), ('MA', true_ma, pred_ma)]):
        ax = axes[row, j]
        ax.scatter(true[:, j], pred[:, j], alpha=0.15, s=8, color='steelblue')
        lims = [-1, 1]
        ax.plot(lims, lims, 'r--', lw=1, label='y=x')
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_xlabel(f'True {name}[{j}]')
        ax.set_ylabel(f'Predicted {name}[{j}]')
        mask = np.abs(true[:, j]) > threshold
        if mask.sum() > 1:
            corr = np.corrcoef(true[mask, j], pred[mask, j])[0, 1]
            ax.set_title(f'{name}[{j}]  r={corr:.3f}')
        ax.grid(True, alpha=0.3)
        ax.set_aspect('equal')

plt.tight_layout()
plt.show()